get_emebbedding.py

In [ ]:
import gguf
import numpy as np
import csv

# Byte-Level BPE tokenlerini okunabilir hale çevirmek için dönüşüm haritası hazırlama
def _get_byte_decoder():
    bs = (
        list(range(ord("!"), ord("~") + 1)) +
        list(range(ord("¡"), ord("¬") + 1)) +
        list(range(ord("®"), ord("ÿ") + 1))
    )
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)
            n += 1
    return {chr(c): b for c, b in zip(cs, bs)}

bpe_decoder_map = _get_byte_decoder()

def clean_bpe_token(token_utf8_str):
    "BPE sembollerini ham byte formatına alıp tekrar UTF-8 olarak okunaklı metne çevirir."
    try:
        # BPE stringindeki her karakteri orjinal byte değerine dönüştürüyoruz
        raw_bytes = bytes([bpe_decoder_map.get(c, ord(c)) for c in token_utf8_str])
        # Şimdi bunu normal bir UTF-8 metni olarak çözümlüyoruz (Bozuksa kaçış sekansına izin veriyoruz)
        return raw_bytes.decode('utf-8', errors='backslashreplace')
    except Exception:
        return token_utf8_str

def read_gguf_tokens(model_path, output_csv="tokens.csv"):
    print(f"Reading model from: {model_path}")
    
    reader = gguf.GGUFReader(model_path)
    token_field_key = "tokenizer.ggml.tokens"
    
    if token_field_key in reader.fields:
        field = reader.fields[token_field_key]
        parts = field.parts  
        
        try:
            total_tokens = parts[4][0]
        except:
            total_tokens = "Bilinmiyor"
            
        token_byte_arrays = parts[6::2]
        
        print(f"{'='*40}")
        print(f"Modelde toplam {total_tokens} adet token bulundu.")
        print(f"Tokenler '{output_csv}' dosyasına kaydediliyor...")
        print(f"{'='*40}")
        
        with open(output_csv, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["ID", "Token"])
            
            for i, token_array in enumerate(token_byte_arrays):
                token_bytes = bytes(token_array)
                # Önce standart utf-8 string olarak alıyoruz (ör. Ġis)
                bpe_str = token_bytes.decode('utf-8', errors='backslashreplace')
                # Sonra bunu tersine çevirerek normal (okunaklı) bir string'e çeviriyoruz
                human_readable_str = clean_bpe_token(bpe_str)
                writer.writerow([i, human_readable_str])
                
        print(f"\nİşlem bitti! Tüm tokenler '{output_csv}' dosyasına yazıldı.")
            
    else:
        print(f"Hata: GGUF dosyasında '{token_field_key}' bulunamadı.")
        for key in reader.fields.keys():
            if "token" in key.lower():
                print(f"- {key}")

def read_gguf_weights(model_path):
    # İhtiyaç olursa diye burada ağırlık okuma fonksiyonu
    reader = gguf.GGUFReader(model_path)
    for tensor in reader.tensors:
        print(tensor.name, tensor.shape)

if __name__ == '__main__':
    # GGUF dosyanızın yolu
    model_dosya_yolu = "./model_weight/Turkish-Gemma-9b-T1.Q4_K_S.gguf" 
    
    # Bütün tokenleri okuyup tokens.csv dosyasına yazdırmak için
    read_gguf_tokens(model_dosya_yolu, output_csv="tokens.csv")


heatmap 2 ve 3

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import matplotlib
import matplotlib as mpl

import fasttext
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Modeli yükle (Bu işlem biraz zaman alabilir)
model = fasttext.load_model("cc.tr.300.bin")

# Karşılaştırmak istediğimiz kelime çiftleri
words = [
    "gözlük", "gözlükçü", 
    "kitap", "kitapçı", 
    "çiçek", "çiçekçi", 
    "saat", "saatçi",
    "yol", "yolcu"
]

# Kelimelerin vektörlerini alalım
vectors = np.array([model[w][:15] for w in words])
farmers = [ str(w) for w in range(15)]

fig, ax = plt.subplots()
im = ax.imshow(vectors)

# Show all ticks and label them with the respective list entries
ax.set_xticks(range(len(farmers)), labels=farmers, rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(len(words)), labels=words)

# Loop over data dimensions and create text annotations.
for i in range(len(words)):
    for j in range(len(farmers)):
        text = ax.text(j, i, vectors[i, j],
                       ha="center", va="center", color="w")

ax.set_title("Harvest of local farmers (in tons/year)")
fig.tight_layout()
plt.show()

In [ ]:
# Buranın bana öğrettiği sonuç: Aslında belli indexlerin 
# kelime eklerine direkt bir ilişkisi olmadığı. Yani kelimeden kelimeye index değişebilir. 




import matplotlib.pyplot as plt
import numpy as np

import matplotlib
import matplotlib as mpl





import fasttext
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

warnings.filterwarnings("ignore")

# Modeli yükle (Bu işlem biraz zaman alabilir)
model = fasttext.load_model("cc.tr.300.bin")

# Karşılaştırmak istediğimiz kelime çiftleri
words = [
    "gözlük", "gözlükçü", # [ 29 295 290 144  54 244  91 226 154 124  46 225 203  40 279]
    "kitap", "kitapçı", # [258  44 273 229 144 298  43  86  32  71 187 133  98 153  11]
    "çiçek", "çiçekçi", 
    "saat", "saatçi",
    "yol", "yolcu"
]

# Kelimelerin vektörlerini alalım
vectors = np.array([model[w] for w in words])

# Burada bana iki tane vektör gelecek. Ben ilk önce iki vektörün tüm 
# elemanlarının, eleman elaman farkını alayım. Farkın en az olduğu ilk 15 indexin 
# numarasını alayım.

# if len(words) != 2:
#     # hata fırlat
#     raise ValueError("Kelime sayısı 2 olmalı")

for i in range(0, len(words)-1, 2):
    word1 = words[i + 1]
    word2 = words[i]

    diff = model[word1] - model[word2]
    # burada sıralama yaparken en başta belirlenen index numaralarını kaybetmemiz lazım.
    # np.argsort() kullanırken indexleri kaybetmemek için np.argsort(diff, kind="stable") kullanabiliriz.
    # diff = np.argsort(diff, kind="stable")[::-1] # büyükten küçüğe doğru sırala
    diff = np.argsort(diff, kind="stable")
    
    diff = diff[:15]

    print(word1, word2, diff)

    

# Büyükten küçüğe sıralama sonucu
# gözlük gözlükçü   [242  24 220 214 257   3 275 217 278 239  61  20 255 212 120]
# kitap kitapçı     [212  20 221  24 241  47 245 206  50 138 132 122 198 249 214]
# çiçek çiçekçi     [ 24  51  42 214 262  84 144 218 251  67 243 142 161  81 138]
# saat saatçi       [214  39  42 105 279 187 190 148 179 285  90  57  61  28 242]
# yol yolcu         [214  67  26  65 160 246 129 261 171  37 292  42 220  88 137]

# küçükten büyüğe
# gözlük gözlükçü    [ 29 295 290 144  54 244  91 226 154 124  46 225 203  40 279]
# kitap kitapçı      [258  44 273 229 144 298  43  86  32  71 187 133  98 153  11]
# çiçek çiçekçi      [169  98  39   7 195 196 293  78 269  34 150 229 288 213 181]
# saat saatçi        [ 79 195 209 133 154  92 215 275 137  44 231 276 189  75 264]
# yol yolcu          [ 49 112 281 259 248 285  38 122  25 170 162 198 207  40 182]


#########

# büyükten küçüğe
# gözlük gözlükçü     [242  24 220 257 278 239  20 255 120  45 108 143 222  15 167]
# kitap kitapçı       [ 20 221  24 241  47  50 138 132 249  17 231 276 215 145 242]
# çiçek çiçekçi       [ 24  42  84 218 251 161 138 223 146 272  70 241 133 156 261]
# saat saatçi         [ 42 279 187 285  90 242 100 130 239 104 217 241  19 164 197]
# yol yolcu           [ 26 160 246 129 261 292  42 220 137 133 189 142 269 116 277]


# gözlük gözlükçü     [214   3 275 217  61 212   6  34 230 243  39 151 114 105 263]
# kitap kitapçı       [212 245 206 122 198 214 266  63  34 112 252 275 277  64 154]
# çiçek çiçekçi       [ 51 214 262 144  67 243 142  81  61  28 248  90   4 113 234]
# saat saatçi         [214  39 105 190 148 179  57  61  28 167  65 180 236 126 124]
# yol yolcu           [214  67  65 171  37  88  27 191  13  61 187 208 230 155 113]

#########



# gözlükçü gözlük    [242  24 220 257 278 239  20 255 120  45 108 143 222  15 167]
# kitapçı kitap      [ 20 221  24 241  47  50 138 132 249  17 231 276 215 145 242]
# çiçekçi çiçek      [ 24  42  84 218 251 161 138 223 146 272  70 241 133 156 261]
# saatçi saat        [ 42 279 187 285  90 242 100 130 239 104 217 241  19 164 197]
# yolcu yol          [ 26 160 246 129 261 292  42 220 137 133 189 142 269 116 277]


# gözlükçü gözlük       [214   3 275 217  61 212   6  34 230 243  39 151 114 105 263]
# kitapçı kitap         [212 245 206 122 198 214 266  63  34 112 252 275 277  64 154]
# çiçekçi çiçek         [ 51 214 262 144  67 243 142  81  61  28 248  90   4 113 234]
# saatçi saat           [214  39 105 190 148 179  57  61  28 167  65 180 236 126 124]
# yolcu yol             [214  67  65 171  37  88  27 191  13  61 187 208 230 155 113]